# Genotype–Phenotype Heterogeneity Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Dataset @id: {metadata['@id']}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Keywords: {', '.join(metadata.keywords)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant datasets, each entity (record set, field, column) is referenced by its unique `@id` field.

In [ ]:
# Discover record sets and their fields
record_sets = dataset.record_sets

print("Record Sets in the dataset:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']} | Name: {rs.get('name', '(no name)')} | Description: {rs.get('description', '(no description)')}")

record_set_ids = [rs['@id'] for rs in record_sets]

print("\nFields for each RecordSet:")
for rs in record_sets:
    fields = rs.get('field', [])
    print(f"\nRecordSet @id: {rs['@id']}")
    for field in fields:
        print(f"  Field @id: {field.get('@id', '(no id)')} | Name: {field.get('name', '(no name)')} | DataType: {field.get('dataType', '(unknown)')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
Use the record set and field `@id`s from the overview above.

Below, we extract the main three tables described in the dataset metadata.

In [ ]:
# List the record_set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show columns for each extracted record set
for rs_id in record_set_ids:
    print(f"\nDataFrame for RecordSet @id: {rs_id}")
    print(f"Columns: {dataframes[rs_id].columns.tolist()}")
    display(dataframes[rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric fields, and grouping data. Use field `@id`s for referencing columns.

In [ ]:
# Example: Filter and normalize numeric field in the first record set
first_rs_id = record_set_ids[0]
df = dataframes[first_rs_id]

# Get numeric fields with their @id
numeric_fields = [field['@id'] for field in filter(lambda f: f.get('dataType', '').lower() in ['integer', 'float', 'number'], dataset.record_sets[0].get('field', []))]
print(f"Numeric fields: {numeric_fields}")

# Choose first numeric field for analysis (example usage)
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    if numeric_field_id in df.columns:
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group data by a categorical field (if any string fields present)
        string_fields = [field['@id'] for field in filter(lambda f: f.get('dataType', '').lower() == 'text', dataset.record_sets[0].get('field', []))]
        if string_fields:
            group_field_id = string_fields[0]
            if group_field_id in filtered_df.columns:
                grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
                print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
                display(grouped.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the selected numeric field for the first record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields and numeric_field_id in df.columns:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of '{numeric_field_id}' in RecordSet {first_rs_id}")
    plt.show()

    # Correlation with another numeric field if present
    if len(numeric_fields) > 1 and numeric_fields[1] in df.columns:
        plt.figure(figsize=(6, 4))
        sns.scatterplot(x=df[numeric_fields[0]], y=df[numeric_fields[1]])
        plt.xlabel(numeric_fields[0])
        plt.ylabel(numeric_fields[1])
        plt.title(f"Scatter between {numeric_fields[0]} and {numeric_fields[1]}")
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, overview, extraction, EDA, and visualization of the FAIR^2 dataset using `mlcroissant`.

- Entities (record sets, fields, columns) were referenced using their unique `@id` throughout.
- With only three patients, data analysis is limited in scope but provides clinical research insights.
- The `mlcroissant` library helps structure data exploration systematically for FAIR datasets.